# 03 · Agentic Experiment Analysis

Load results produced by the autonomous NAS agent (`agents/nas_agent.py`),
inspect the agent memory, ablation statistics and sweet-spot models.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import pandas as pd
import matplotlib.pyplot as plt

EXP_DIR = Path('../experiments/agentic_run')  # change if needed
print('Looking in:', EXP_DIR.resolve())

In [ ]:
# Load Pareto front
pareto_path = EXP_DIR / 'pareto_front.json'
if pareto_path.exists():
    with open(pareto_path) as f:
        front = json.load(f)
    df = pd.DataFrame(front)
    display(df.head(10))
else:
    print('No Pareto front found. Run the agent first:')
    print('  python -m agents.nas_agent --exp-name agentic_run --max-steps 5')

In [ ]:
if 'df' in dir() and len(df):
    fig, ax = plt.subplots(figsize=(8, 5))
    for attn, grp in df.groupby('attn_type'):
        ax.scatter(grp['params_M'], grp['accuracy'], label=attn, s=70, alpha=0.85, edgecolors='k')
    ax.set_xlabel('Parameters (M)')
    ax.set_ylabel('Accuracy')
    ax.set_title('Pareto front coloured by attention type')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

out_dir = Path("../experiments/agentic_run/figures")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "pareto_front_agentic.png", dpi=200, bbox_inches="tight")
print(f"Saved agentic Pareto plot to: {out_dir / "pareto_front_agentic.png"}")


In [ ]:
# Agent memory
mem_path = EXP_DIR / 'agent_memory.json'
if mem_path.exists():
    with open(mem_path) as f:
        memory = json.load(f)
    print(f'Agent took {len(memory)} steps:\n')
    for e in memory:
        print(f"  Step {e['step']}: {e['action']['name']:15s} → {e['outcome'][:60]}")
else:
    print('No agent memory yet.')

In [ ]:
# Analysis / ablation if present
for name in ['analysis.json', 'ablation.json']:
    p = EXP_DIR / name
    if p.exists():
        print(f'\n=== {name} ===')
        with open(p) as f:
            data = json.load(f)
        print(json.dumps(data, indent=2)[:800])

In [ ]:
# Search history overview
hist_path = EXP_DIR / "search_history.json"
if hist_path.exists():
    with open(hist_path) as f:
        history = json.load(f)
    if history:
        gens = [entry["generation"] for entry in history]
        hv = [entry["hypervolume"] for entry in history]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(gens, hv, "o-", color="darkorange")
        ax.set_xlabel("Generation")
        ax.set_ylabel("Hypervolume")
        ax.set_title("Agentic search hypervolume over time")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        out_dir = EXP_DIR / "figures"
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / "search_history_hypervolume.png", dpi=200, bbox_inches="tight")
        plt.show()
        print(f"Saved search history plot to: {out_dir / "search_history_hypervolume.png"}")
    else:
        print("Search history is empty.")
else:
    print("No search history found; run the agent first.")


In [ ]:
# Agent action summary
mem_path = EXP_DIR / "agent_memory.json"
if mem_path.exists():
    with open(mem_path) as f:
        memory = json.load(f)
    actions = [step["action"]["name"] for step in memory]
    if actions:
        counts = pd.Series(actions).value_counts()
        fig, ax = plt.subplots(figsize=(8, 4))
        counts.plot(kind="bar", ax=ax, color="steelblue")
        ax.set_title("Agent action distribution")
        ax.set_xlabel("Action")
        ax.set_ylabel("Count")
        plt.xticks(rotation=20)
        plt.tight_layout()
        out_dir = EXP_DIR / "figures"
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / "agent_action_summary.png", dpi=200, bbox_inches="tight")
        plt.show()
        print(f"Saved action summary to: {out_dir / "agent_action_summary.png"}")
    else:
        print("No agent actions to summarize.")
else:
    print("No agent memory available yet; run the agent to generate an action summary.")


## How to re-run the agent

```bash
cd LowRes_EffTransNAS
python -m agents.nas_agent \
    --goal "models under 15M params with accuracy > 85" \
    --exp-name agentic_run \
    --max-steps 6 \
    --pop-size 20 \
    --generations 8
```

Then re-execute this notebook to inspect the new artefacts.